# Eine interaktive Weltkarte mit `ipyleaflet`

Am Ende dieses Notebooks steht eine Weltkarte: Fährst du mit der Maus über ein Land, zeichnet ein Diagramm daneben, wie sich dort Einkommen, Bevölkerung oder Lebenserwartung zwischen 1800 und 2009 entwickelt haben.

Drei Bibliotheken arbeiten dafür zusammen: `ipyleaflet` zeichnet die Karte, `bqplot` das Diagramm, `ipywidgets` steuert beides.

In [ ]:
%pip install -q bqplot ipyleaflet

## Bibliotheken importieren

In [ ]:
import os
from urllib.request import urlopen
import json
from io import StringIO
from datetime import datetime

import numpy as np
import pandas as pd

from js import fetch

from ipywidgets import Dropdown

from bqplot import Lines, Figure, LinearScale, DateScale, Axis

from ipyleaflet import Map, GeoJSON, WidgetControl

## Die Daten holen

Die Zahlen stammen aus dem Gapminder-Datensatz und liegen als JSON-Datei im Netz. `fetch` ist die Download-Funktion des Browsers – `await` wartet, bis die Antwort da ist.

In [ ]:
URL = "https://raw.githubusercontent.com/jupyter-widgets/ipyleaflet/master/examples/nations.json"

res = await fetch(URL)
text = await res.text()

data = pd.read_json(StringIO(text))

## Die Daten aufbereiten

Nicht für jedes Land gibt es Werte für jedes Jahr. `clean_data` wirft Länder mit zu wenigen Messwerten heraus, `extrap_data` rechnet die übrigen auf einen Wert pro Jahr hoch – dazwischen wird linear interpoliert.

In [ ]:
def clean_data(data):
    for column in ['income', 'lifeExpectancy', 'population']:
        data = data.drop(data[data[column].apply(len) <= 4].index)
    return data

def extrap_interp(data):
    data = np.array(data)
    x_range = np.arange(1800, 2009, 1.)
    y_range = np.interp(x_range, data[:, 0], data[:, 1])
    return y_range

def extrap_data(data):
    for column in ['income', 'lifeExpectancy', 'population']:
        data[column] = data[column].apply(extrap_interp)
    return data

In [ ]:
data = clean_data(data)
data = extrap_data(data)

Ein Blick in die fertige Tabelle:

In [ ]:
data

## Das Diagramm vorbereiten

Eine Achse in `bqplot` besteht aus einer *Skala* (welcher Wertebereich?) und einer *Achse* (wie wird sie beschriftet?). Hier ist die x-Achse eine Zeitachse von 1800 bis 2009.

In [ ]:
date_start = datetime(1800, 12, 31)
date_end = datetime(2009, 12, 31)

date_scale = DateScale(min=date_start, max=date_end)

In [ ]:
date_data = pd.date_range(start=date_start, end=date_end, freq='YE', normalize=True)

Als Startwert zeigt das Diagramm ein Land; per Maus wird das gleich austauschbar.

In [ ]:
country_name = 'Angola'
data_name = 'income'

x_data = data[data.name == country_name][data_name].values[0]

Daraus bauen wir das Liniendiagramm:

In [ ]:
x_scale = LinearScale()

lines = Lines(x=date_data, y=x_data, scales={'x': date_scale, 'y': x_scale})

ax_x = Axis(label='Year', scale=date_scale, num_ticks=10, tick_format='%Y')
ax_y = Axis(label=data_name.capitalize(), scale=x_scale, orientation='vertical', side='left')

figure = Figure(axes=[ax_x, ax_y], title=country_name, marks=[lines], animation_duration=500,
                layout={'max_height': '250px', 'max_width': '400px'})

Diese Funktion stellt das Diagramm auf ein anderes Land oder eine andere Größe um. Sie wird später bei jeder Mausbewegung aufgerufen.

In [ ]:
def update_figure(country_name, data_name):
    try:
        lines.y = data[data.name == country_name][data_name].values[0]
        ax_y.label = data_name.capitalize()
        figure.title = country_name
    except IndexError:
        pass

## Die Karte

Die Ländergrenzen kommen als GeoJSON – ein Format, das Umrisse als Listen von Koordinaten beschreibt.

In [ ]:
URL = "https://raw.githubusercontent.com/jupyter-widgets/ipyleaflet/master/examples/countries.geo.json"

res = await fetch(URL)
text = await res.text()

countries = json.loads(text)

Karte anlegen und die Grenzen als Ebene darüberlegen:

In [ ]:
m = Map(zoom=3)

geo = GeoJSON(data=countries, style={'fillColor': 'white', 'weight': 0.5}, hover_style={'fillColor': '#1f77b4'}, name='Countries')
m.add_layer(geo)

## Karte und Diagramm verbinden

`WidgetControl` heftet das Diagramm in eine Ecke der Karte. `on_hover` sorgt dafür, dass bei jeder Mausbewegung über ein Land das Diagramm neu gezeichnet wird.

In [ ]:
widget_control1 = WidgetControl(widget=figure, position='bottomright')

m.add_control(widget_control1)

def on_hover(event, feature, **kwargs):
    global country_name

    country_name = feature['properties']['name']
    update_figure(country_name, data_name)

geo.on_hover(on_hover)

## Ein Auswahlfeld für die Größe

Mit dem Dropdown lässt sich umschalten zwischen Einkommen, Bevölkerung und Lebenserwartung.

In [ ]:
dropdown = Dropdown(
    options=['income', 'population', 'lifeExpectancy'],
    value=data_name,
    description='Plotting:'
)

def on_click(change):
    global data_name

    data_name = change['new']
    update_figure(country_name, data_name)

dropdown.observe(on_click, 'value')

widget_control2 = WidgetControl(widget=dropdown, position='bottomleft')

m.add_control(widget_control2)

## Fertig

Führe die Zelle aus und fahre mit der Maus über die Länder. Unten links wählst du aus, welche Größe das Diagramm zeigt.

In [ ]:
m